<a href="https://colab.research.google.com/github/Wezz-git/AI-samples/blob/main/Advanced_SQL_Feature_Engineering_w_XGBoost_Risk_Model_Execution_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**The Focus:** Database Connectivity and Complex Feature Creation.

**Project:** Calculating Rolling Risk Metrics via SQL.

**Goal:** Move beyond basic Python cleaning. Write code that executes advanced SQL (specifically Window Functions) to create the complex, time-dependent features required for risk and modeling.

**Skill:** Mastering SQL Window Functions, which are non-negotiable for generating features like rolling averages, lags, and customer rankings from large databases.

Core Concept: SQL Window Functions

In Python (pandas), calculate a rolling average like this: df['col'].rolling(window=30).mean().

In SQL, use of Window Function (often AVG() OVER (...)). This is powerful because it allows to calculate a metric (like an average) across a specific window of related rows (like the last 30 days) without writing complex subqueries or joins.

Set up & Moch data creation

In [1]:
import pandas as pd
import numpy as np
import sqlite3

# Create Mock database connection

conn = sqlite3.connect(':memory:')     # :memory: - creates temporary, in-memory database

# Create sample data simulating 10 days of transactions
data = {
    'transaction_date': pd.to_datetime(pd.date_range(start='2025-11-01', periods=10, freq='D')),
    'customer_id': ['CUST001'] * 10,
    'daily_balance' : [1000, 1050, 1020, 950, 1100, 1200, 1150, 1300, 1250, 1400],
    'is_high_risk' : [0, 0, 0, 1, 0, 0, 0, 1, 0, 0]                                 # 1 meaning risk flag raised
}
df_transactions = pd.DataFrame(data)

# Push pandas DataFrame into SQL database (mock)
df_transactions.to_sql('transactions', conn, index=False, if_exists='replace')

print("--- Raw Data in SQL (Mock) Table ---")
print(pd.read_sql("SELECT * FROM transactions", conn))

--- Raw Data in SQL (Mock) Table ---
      transaction_date customer_id  daily_balance  is_high_risk
0  2025-11-01 00:00:00     CUST001           1000             0
1  2025-11-02 00:00:00     CUST001           1050             0
2  2025-11-03 00:00:00     CUST001           1020             0
3  2025-11-04 00:00:00     CUST001            950             1
4  2025-11-05 00:00:00     CUST001           1100             0
5  2025-11-06 00:00:00     CUST001           1200             0
6  2025-11-07 00:00:00     CUST001           1150             0
7  2025-11-08 00:00:00     CUST001           1300             1
8  2025-11-09 00:00:00     CUST001           1250             0
9  2025-11-10 00:00:00     CUST001           1400             0


Advanced SQL Query (window functions)



In [2]:
sql_query = """
SELECT
    transaction_date,
    daily_balance,
    -- THE WINDOW FUNCTION FEATURE IS CREATED HERE
    AVG(daily_balance) OVER (
        PARTITION BY customer_id
        ORDER BY transaction_date
        -- Window includes the current row and the 2 preceding rows (total 3 days)
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_3day_avg,
    is_high_risk
FROM
    transactions
ORDER BY
    transaction_date;
"""

# Execute query and load results to pandas DF
df_features = pd.read_sql(sql_query, conn)

print("\n--- Calculated Features in SQL ---")
print(df_features)


--- Calculated Features in SQL ---
      transaction_date  daily_balance  rolling_3day_avg  is_high_risk
0  2025-11-01 00:00:00           1000       1000.000000             0
1  2025-11-02 00:00:00           1050       1025.000000             0
2  2025-11-03 00:00:00           1020       1023.333333             0
3  2025-11-04 00:00:00            950       1006.666667             1
4  2025-11-05 00:00:00           1100       1023.333333             0
5  2025-11-06 00:00:00           1200       1083.333333             0
6  2025-11-07 00:00:00           1150       1150.000000             0
7  2025-11-08 00:00:00           1300       1216.666667             1
8  2025-11-09 00:00:00           1250       1233.333333             0
9  2025-11-10 00:00:00           1400       1316.666667             0


look at the rolling_3day_avg column.

Row 1 (Oct 1): The average should be the balance itself (1000.0).

Row 3 (Oct 3): The average should be (1000 + 1050 + 1020) / 3 = 1023.33

Row 10 (Oct 10): The average is the last three days: (1300 + 1250 + 1400) / 3 = 1316.67

Align Data & Prepare for XGBoost



Goal: Use rolling_3day_avg feature created to predict the is_high_risk flag, demonstrating that the rolling average is a strong predictor of future problems.

In [10]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

# Prepare data from df_features
# Target(y) and Features(X)

y = df_features['is_high_risk']
X = df_features[['daily_balance', 'rolling_3day_avg']]

# Split data into train and test sets
# Stratify ensures both classes (0 and 1) are present in test set.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42,
    stratify=y)     # forces split to maintain the ratio of classes

# Train XGBoost model
model_xgb = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)

print('Training XGBoost risk model')
model_xgb.fit(X_train, y_train)

# Evaluate performance
predictions = model_xgb.predict(X_test)
report = classification_report(y_test, predictions, labels=[0, 1], target_names=['Low Risk (0)', 'High Risk (1)'])

print("\n--- Classification Report ---")
print(report)

Training XGBoost risk model

--- Classification Report ---
               precision    recall  f1-score   support

 Low Risk (0)       1.00      1.00      1.00         2
High Risk (1)       0.00      0.00      0.00         0

     accuracy                           1.00         2
    macro avg       0.50      0.50      0.50         2
 weighted avg       1.00      1.00      1.00         2



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [00:38:54] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 

Low Risk (0)	2	- Dad 2 genuinely low-risk samples in the test set.

High Risk (1)	0	CRITICAL FAILURE: Had zero genuinely high-risk samples in the test set.

Total	2	test set only contained 2 samples in total.

The accuracy is 1.00 because the model had two samples (both 'Low Risk 0') and correctly predicted 'Low Risk 0' for both.